# DATA_PREPARE — chuẩn bị dữ liệu

Chạy **một lần duy nhất** trên Colab. Khoảng 30 phút.

```
Zenodo (5.7 GB)  ->  giải nén 13 GB  ->  5 script  ->  cất ~2.7 GB lên Drive
```

Sau đó mọi notebook thí nghiệm chỉ cần giải nén từ Drive, mất 2 phút.

**Không cần GPU.** Runtime → CPU cũng được, đỡ tốn quota.

## 0. Kiểm tra môi trường

In [ ]:
import os
import subprocess

def chay(lenh):
    ket_qua = subprocess.run(lenh, shell=True, capture_output=True, text=True)
    return (ket_qua.stdout + ket_qua.stderr).strip()

print(chay("df -h /content | tail -1"))
print("cần ít nhất 20 GB trống cho zip 5.7 GB + CSV giải nén 13 GB")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy code

Repo MobiVital **không có LICENSE** nên không được chép vào repo mình. Clone
riêng mỗi phiên là cách duy nhất đúng.

In [ ]:
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
!git clone -q https://github.com/nesl/mobivital-public.git /content/UWB_RADAR/external/mobivital
!pip install -q einops
!ls /content/UWB_RADAR

In [ ]:
%cd /content/UWB_RADAR

## 3. Tải dataset từ Zenodo

Dùng `aria2c` chia 16 luồng, không dùng `wget`. Đo thật trên Colab:

```
wget    0.7 MB/s   ->  2.3 gio
aria2c   56 MB/s   ->  2 phut
```

In [ ]:
!apt-get install -qq -y aria2

In [ ]:
!aria2c -x16 -s16 -k5M --console-log-level=warn -d /content -o tripod.zip \
    https://zenodo.org/api/records/15022885/files/tripod.zip/content
!ls -la /content/tripod.zip

## 4. Giải nén — 5.7 GB nén thành 13 GB CSV

In [ ]:
# Trong zip da co san thu muc "tripod/", nen giai nen thang vao data/raw/
# chu KHONG vao data/raw/tripod/ -- khong thi long hai tang.
!mkdir -p data/raw
!unzip -q -o /content/tripod.zip -d data/raw/
!rm /content/tripod.zip
print("số file CSV:", chay("find data/raw -name '*.csv' | wc -l"))

## 5. Gom CSV theo người

`data/raw/tripod/*.csv` (1874 file lẫn lộn) → `data/raw/A/` … `data/raw/L/`

Tên file có dạng `240409_userG_tripod_02_3.csv`, lấy chữ cái ngay sau `user`.

In [ ]:
!python scripts/1_organize_raw.py

## 6. → `by_user/*.npz` — pipeline DEV

Mỗi người một file, gồm 3 mảng:

```
uwb    (n, 1500, 120) complex64   tin hieu radar
gt     (n, 1500)      float32     nhip tho that, da chuan hoa [-1, 1]
files  (n,)                       ten file CSV, de doi chieu ve sau
```

CSV có 1500 dòng × 254 cột. Cột 12–131 là phần thực, 132–251 phần ảo,
cột áp chót là nhịp thở thật. File nào không đủ 1500 dòng thì bỏ.

In [ ]:
!python scripts/2_make_npz.py

## 7. → `mobivital_original/*.npy` — pipeline GỐC

Chạy `prep_breath_final.py` của MobiVital **nguyên bản, 0 dòng sửa**.

Code họ dùng đường dẫn tương đối (`./dataset/mobivital/tripod/`) nên script
này dựng một thư mục tạm có đúng cấu trúc đó bằng symlink, rồi `cd` vào chạy.

File này **chỉ dùng để đối chứng ở bước 8**, không đưa lên Drive.

In [ ]:
!python scripts/3_run_mobivital_prep.py

## 8. Đối chiếu hai pipeline — bước quan trọng nhất

Phải in ra **`Khac nhau nhieu nhat tren 1500 mau: 0.0`**.

Đó là bằng chứng dữ liệu mình tự đọc giống hệt dữ liệu MobiVital dùng. Nhờ nó
mà mọi thí nghiệm sau chỉ cần đọc `by_user/*.npz`, không cần CSV thô nữa.

Không ra 0 thì **dừng lại**, đừng chạy tiếp.

In [ ]:
!python scripts/4_check_data.py

## 9. Cắt cửa sổ để train

```
200 mau vao  ->  25 mau model phai doan,  truot 25  =>  52 cua so moi song
```

Gọi `generate_dataset` của MobiVital, cắt riêng từng người để sau ghép 4 fold
tuỳ ý. Kết quả `data/processed/windows/dev_cv/*.npz`.

In [ ]:
!python scripts/5_make_windows.py

## 10. Cất lên Drive

Chỉ đưa lên hai thứ cần cho các thí nghiệm sau:

| | dùng để | dung lượng |
|---|---|---|
| `windows/dev_cv/` | train | ~250 MB |
| `by_user/` | chấm điểm (cần uwb phức thô) | ~2.4 GB |

**Không đưa lên:**

- `data/raw/` 13 GB — chỉ cần lúc chuẩn bị
- `mobivital_original/` 2.6 GB — chỉ để đối chứng ở bước 8
- `windows/final_train/` 251 MB — chính là 8 file `dev_cv` ghép lại,
  đã kiểm chứng cùng tập cửa sổ, ghép lúc chạy cũng được

In [ ]:
!mkdir -p /content/drive/MyDrive/mobivital
!tar -czf /content/drive/MyDrive/mobivital/windows_dev_cv.tar.gz -C data/processed/windows dev_cv
!tar -cf  /content/drive/MyDrive/mobivital/by_user.tar          -C data/processed by_user

`by_user` dùng `tar` không nén: dữ liệu radar nhiễu nên gzip chỉ giảm 6%
(147 MB → 138 MB, đã đo) mà tốn thêm vài phút. Không đáng.

In [ ]:
print(chay("ls -la /content/drive/MyDrive/mobivital"))
print()
print("Drive đang dùng:", chay("du -sh /content/drive/MyDrive/mobivital | cut -f1"))

## Xong

Từ giờ mọi notebook thí nghiệm bắt đầu bằng:

```python
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!git clone -q https://github.com/nesl/mobivital-public.git external/mobivital
!pip install -q einops

!mkdir -p data/processed
!tar -xzf /content/drive/MyDrive/mobivital/windows_dev_cv.tar.gz -C data/processed/
!tar -xf  /content/drive/MyDrive/mobivital/by_user.tar          -C data/processed/

!ln -s /content/drive/MyDrive/mobivital/runs runs
```

Mất khoảng 2 phút. `runs/` trỏ thẳng vào Drive nên Colab ngắt phiên vẫn giữ
được checkpoint.